Merge the data

In [2]:
import os
import pandas as pd

# Define the input and output directories
input_directory = '/mnt/scratch_lustre/barthelx/Masrur/Projects/Data_imputation/raw_data'
output_directory = '/mnt/scratch_lustre/barthelx/Masrur/Projects/Data_imputation/Modified_Data'

# Create the output directory if it doesn't exist
os.makedirs(output_directory, exist_ok=True)

# Function to extract study site from filename
def extract_study_site(filename):
    parts = filename.split('_')
    study_site = parts[-3]  # Assuming the study site is the third word from the end (ignoring .csv)
    return study_site

# Function to merge CSV files by study site
def merge_csv_by_study_site(input_directory, output_directory):
    # Dictionary to hold data for each study site
    data_dict = {}

    # Iterate over each file in the input directory
    for file in os.listdir(input_directory):
        if file.endswith(".csv"):
            study_site = extract_study_site(file)
            file_path = os.path.join(input_directory, file)

            # Read the CSV file
            df = pd.read_csv(file_path)
            df['datetime'] = pd.to_datetime(df['datetime'], utc=True)  # Ensure the 'datetime' is in UTC

            # Add the data to the dictionary
            if study_site not in data_dict:
                data_dict[study_site] = []
            data_dict[study_site].append(df)

    # Merge and save data for each study site
    for study_site, data_list in data_dict.items():
        # Concatenate the dataframes
        merged_df = pd.concat(data_list, ignore_index=True)

        # Sort the dataframe by datetime
        merged_df.sort_values(by='datetime', inplace=True)

        # Rename columns and extract study site
        study_site = rename_columns(merged_df)

        # Get start and end dates
        start_date = merged_df['datetime'].iloc[0].strftime('%Y-%m-%d')
        end_date = merged_df['datetime'].iloc[-1].strftime('%Y-%m-%d')

        # Save the merged dataframe to the output directory
        output_file_path = os.path.join(output_directory, f'AQMS_{study_site}_{start_date}_{end_date}_data.csv')
        merged_df.to_csv(output_file_path, index=False)
        print(f'Saved merged and modified data for {study_site} to {output_file_path}')

# Function to rename columns by extracting the variable name
def rename_columns(df):
    renamed_columns = {}
    study_site = None
    for col in df.columns:
        if col != 'datetime':
            parts = col.split('_')
            if len(parts) > 1:
                var_name = parts[0]
                renamed_columns[col] = var_name
                if not study_site:
                    study_site = parts[1]  # Assuming all columns have the same study site
    df.rename(columns=renamed_columns, inplace=True)
    return study_site

# Merge CSV files by study site and save directly to the output directory
merge_csv_by_study_site(input_directory, output_directory)

print("Merging, renaming columns, and saving complete.")


Saved merged and modified data for LIVERPOOL to /mnt/scratch_lustre/barthelx/Masrur/Projects/Data_imputation/Modified_Data/AQMS_LIVERPOOL_2018-12-31_2024-07-16_data.csv
Saved merged and modified data for WOLLONGONG to /mnt/scratch_lustre/barthelx/Masrur/Projects/Data_imputation/Modified_Data/AQMS_WOLLONGONG_2018-12-31_2024-07-16_data.csv
Saved merged and modified data for BATHURST to /mnt/scratch_lustre/barthelx/Masrur/Projects/Data_imputation/Modified_Data/AQMS_BATHURST_2018-12-31_2024-07-16_data.csv
Saved merged and modified data for LIDCOMBE to /mnt/scratch_lustre/barthelx/Masrur/Projects/Data_imputation/Modified_Data/AQMS_LIDCOMBE_2020-04-15_2024-07-16_data.csv
Saved merged and modified data for NEWCASTLE to /mnt/scratch_lustre/barthelx/Masrur/Projects/Data_imputation/Modified_Data/AQMS_NEWCASTLE_2018-12-31_2024-07-16_data.csv
Saved merged and modified data for WAGGA to /mnt/scratch_lustre/barthelx/Masrur/Projects/Data_imputation/Modified_Data/AQMS_WAGGA_2018-12-31_2024-07-16_data.

Checking wehather the date and time is continous 

In [1]:
import os
import pandas as pd
from datetime import timedelta

# Define the input and output directories
input_directory = '/mnt/scratch_lustre/barthelx/Masrur/Projects/Data_imputation/Modified_Data'
output_directory = '/mnt/scratch_lustre/barthelx/Masrur/Projects/Data_imputation/sorted_Continuous_Data'
report_directory = '/mnt/scratch_lustre/barthelx/Masrur/Projects/Data_imputation/Summery_results/report'

# Create the output and report directories if they don't exist
os.makedirs(output_directory, exist_ok=True)
os.makedirs(report_directory, exist_ok=True)

# Function to ensure continuous datetime records
def ensure_continuous_datetime(df):
    df['datetime'] = pd.to_datetime(df['datetime'])
    df = df.sort_values(by='datetime')
    
    # Create a date range with all timestamps
    start_date = df['datetime'].min()
    end_date = df['datetime'].max()
    full_range = pd.date_range(start=start_date, end=end_date, freq='H')
    
    # Reindex dataframe to include all hours in the range
    df = df.set_index('datetime').reindex(full_range).reset_index()
    df.rename(columns={'index': 'datetime'}, inplace=True)
    
    return df

# Function to process each CSV file
def process_csv_files(input_directory, output_directory, report_directory):
    for filename in os.listdir(input_directory):
        if filename.endswith('.csv'):
            file_path = os.path.join(input_directory, filename)

            # Load the CSV file into a DataFrame
            df = pd.read_csv(file_path)

            # Ensure continuous datetime
            continuous_df = ensure_continuous_datetime(df)

            # Check for added datetime entries
            added_datetimes = continuous_df[continuous_df['datetime'].isin(df['datetime']) == False]

            # Save the continuous DataFrame to a new CSV file
            continuous_file_path = os.path.join(output_directory, filename)
            continuous_df.to_csv(continuous_file_path, index=False)
            print(f"Saved continuous data for {filename} to {continuous_file_path}")

            # Save the report of added datetime entries
            if not added_datetimes.empty:
                report_file_path = os.path.join(report_directory, f'Report_{filename}')
                added_datetimes.to_csv(report_file_path, index=False)
                print(f"Saved report of added datetimes for {filename} to {report_file_path}")

process_csv_files(input_directory, output_directory, report_directory)

print("Processing complete. Continuous data and reports have been saved.")


Saved continuous data for AQMS_NEWCASTLE_2018-12-31_2024-07-16_data.csv to /mnt/scratch_lustre/barthelx/Masrur/Projects/Data_imputation/sorted_Continuous_Data/AQMS_NEWCASTLE_2018-12-31_2024-07-16_data.csv
Saved continuous data for AQMS_PARRAMATTA_2018-12-31_2024-07-16_data.csv to /mnt/scratch_lustre/barthelx/Masrur/Projects/Data_imputation/sorted_Continuous_Data/AQMS_PARRAMATTA_2018-12-31_2024-07-16_data.csv
Saved continuous data for AQMS_LIDCOMBE_2020-04-15_2024-07-16_data.csv to /mnt/scratch_lustre/barthelx/Masrur/Projects/Data_imputation/sorted_Continuous_Data/AQMS_LIDCOMBE_2020-04-15_2024-07-16_data.csv
Saved report of added datetimes for AQMS_LIDCOMBE_2020-04-15_2024-07-16_data.csv to /mnt/scratch_lustre/barthelx/Masrur/Projects/Data_imputation/Summery_results/report/Report_AQMS_LIDCOMBE_2020-04-15_2024-07-16_data.csv
Saved continuous data for AQMS_LIVERPOOL_2018-12-31_2024-07-16_data.csv to /mnt/scratch_lustre/barthelx/Masrur/Projects/Data_imputation/sorted_Continuous_Data/AQMS_L

In [ ]:
import os
import pandas as pd

# Define the input and output directories
input_directory = '/mnt/scratch_lustre/barthelx/Masrur/Projects/Data_imputation/raw_data'
output_directory = '/mnt/scratch_lustre/barthelx/Masrur/Projects/Data_imputation/Modified_Data'

# Create the output directory if it doesn't exist
os.makedirs(output_directory, exist_ok=True)

# Function to extract study site from filename
def extract_study_site(filename):
    parts = filename.split('_')
    study_site = parts[-3]  # Assuming the study site is the third word from the end (ignoring .csv)
    return study_site

# Function to merge CSV files by study site
def merge_csv_by_study_site(input_directory, output_directory):
    # Dictionary to hold data for each study site
    data_dict = {}

    # Iterate over each file in the input directory
    for file in os.listdir(input_directory):
        if file.endswith(".csv"):
            study_site = extract_study_site(file)
            file_path = os.path.join(input_directory, file)

            # Read the CSV file
            df = pd.read_csv(file_path)
            df['datetime'] = pd.to_datetime(df['datetime'], utc=True)  # Ensure the 'datetime' is in UTC

            # Add the data to the dictionary
            if study_site not in data_dict:
                data_dict[study_site] = []
            data_dict[study_site].append(df)

    # Merge and save data for each study site
    for study_site, data_list in data_dict.items():
        # Concatenate the dataframes
        merged_df = pd.concat(data_list, ignore_index=True)

        # Sort the dataframe by datetime
        merged_df.sort_values(by='datetime', inplace=True)

        # Rename columns and extract study site
        study_site = rename_columns(merged_df)

        # Get start and end dates
        start_date = merged_df['datetime'].iloc[0].strftime('%Y-%m-%d')
        end_date = merged_df['datetime'].iloc[-1].strftime('%Y-%m-%d')

        # Save the merged dataframe to the output directory
        output_file_path = os.path.join(output_directory, f'AQMS_{study_site}_{start_date}_{end_date}_data.csv')
        merged_df.to_csv(output_file_path, index=False)
        print(f'Saved merged and modified data for {study_site} to {output_file_path}')

# Function to rename columns by extracting the variable name
def rename_columns(df):
    renamed_columns = {}
    study_site = None
    for col in df.columns:
        if col != 'datetime':
            parts = col.split('_')
            if len(parts) > 1:
                var_name = parts[0]
                renamed_columns[col] = var_name
                if not study_site:
                    study_site = parts[1]  # Assuming all columns have the same study site
    df.rename(columns=renamed_columns, inplace=True)
    return study_site

# Merge CSV files by study site and save directly to the output directory
merge_csv_by_study_site(input_directory, output_directory)

print("Merging, renaming columns, and saving complete.")


In [2]:
import os
import pandas as pd

# Define the input directory containing the CSV files
input_directory = '/home/ahmedmas/Projects/Data_imputation/Modified_Data'

# Desired study site and new start date
desired_study_site = 'LIDCOMBE'
new_start_date = pd.Timestamp('2020-03-26 07:00:00+00:00')

# Function to extract study site from filename
def extract_study_site(filename):
    parts = filename.split('_')
    study_site = parts[1]  # Assuming the study site is the second word in the filename
    return study_site

# Traverse the directory and process each CSV file
for filename in os.listdir(input_directory):
    if filename.endswith('.csv'):
        study_site = extract_study_site(filename)
        
        if study_site == desired_study_site:
            filepath = os.path.join(input_directory, filename)

            # Load the CSV file into a DataFrame
            df = pd.read_csv(filepath)

            # Convert datetime column to datetime type
            df['datetime'] = pd.to_datetime(df['datetime'])

            # Calculate the time delta to adjust the start date
            original_start_date = df['datetime'].min()
            time_delta = new_start_date - original_start_date

            # Adjust the datetime column
            df['datetime'] = df['datetime'] + time_delta

            # Save the modified DataFrame to a new CSV file with the same name in the same directory
            df.to_csv(filepath, index=False)
            print(f"Modified and saved file: {filepath}")

print("Processing complete.")


Modified and saved file: /home/ahmedmas/Projects/Data_imputation/Modified_Data/AQMS_LIDCOMBE_2020-04-15_2024-07-16_data.csv
Processing complete.


##% of negative value and replacing with zero

In [1]:
import pandas as pd
import os

# Define the input and output directories
input_directory = "/home/ahmedmas/Projects/Data_imputation/Modified_Data_with_10_percent_missing_PM2.5"
output_directory = "/home/ahmedmas/Projects/Data_imputation/Data_After_neg"

# Create the output directory if it doesn't exist
os.makedirs(output_directory, exist_ok=True)

# Function to process each file
def process_file(file_path, output_directory):
    # Read the CSV file
    df = pd.read_csv(file_path)

    # Calculate the percentage of negative values in PM2.5
    pm25_neg_percentage = (df['PM2.5'] < 0).mean() * 100

    # Calculate the percentage of negative values in PM10
    pm10_neg_percentage = (df['PM10'] < 0).mean() * 100

    # Replace negative values with zero
    df['PM2.5'] = df['PM2.5'].apply(lambda x: 0 if x < 0 else x)
    df['PM10'] = df['PM10'].apply(lambda x: 0 if x < 0 else x)

    # Define the output file path
    output_file_path = os.path.join(output_directory, os.path.basename(file_path))

    # Save the processed file in the output directory
    df.to_csv(output_file_path, index=False)

    return pm25_neg_percentage, pm10_neg_percentage

# Process all files in the directory
for file_name in os.listdir(input_directory):
    if file_name.endswith('.csv'):
        file_path = os.path.join(input_directory, file_name)
        pm25_neg_pct, pm10_neg_pct = process_file(file_path, output_directory)
        print(f"Processed {file_name}: PM2.5 Negative Value Percentage: {pm25_neg_pct:.2f}%, PM10 Negative Value Percentage: {pm10_neg_pct:.2f}%")


Processed AQMS_ARMIDALE_2018-12-31_2024-07-16_data.csv: PM2.5 Negative Value Percentage: 7.81%, PM10 Negative Value Percentage: 4.01%
Processed AQMS_BATHURST_2018-12-31_2024-07-16_data.csv: PM2.5 Negative Value Percentage: 8.08%, PM10 Negative Value Percentage: 1.48%
Processed AQMS_LIDCOMBE_2020-04-15_2024-07-16_data.csv: PM2.5 Negative Value Percentage: 9.47%, PM10 Negative Value Percentage: 0.82%
Processed AQMS_LIVERPOOL_2018-12-31_2024-07-16_data.csv: PM2.5 Negative Value Percentage: 5.98%, PM10 Negative Value Percentage: 0.78%
Processed AQMS_NEWCASTLE_2018-12-31_2024-07-16_data.csv: PM2.5 Negative Value Percentage: 7.39%, PM10 Negative Value Percentage: 0.70%
Processed AQMS_PARRAMATTA_2018-12-31_2024-07-16_data.csv: PM2.5 Negative Value Percentage: 3.87%, PM10 Negative Value Percentage: 0.84%
Processed AQMS_WAGGA_2018-12-31_2024-07-16_data.csv: PM2.5 Negative Value Percentage: 9.13%, PM10 Negative Value Percentage: 0.89%
Processed AQMS_WOLLONGONG_2018-12-31_2024-07-16_data.csv: PM2